# Duka Electronics — Half-Year Sales Exploration (Jan–Jun 2026)

## Executive Summary
- Duka Electronics recorded 1,000 orders across five branches during the first half of 2026 (January–June 2026).
- The Nairobi branch leads branch performance with 400 orders and a 36.4% share of total revenue, nearly double second-placed Mombasa.
- Laptops generate KES 41,535,000 in revenue, more than double the next highest-earning product.
- Online orders (320 orders) achieve an average order value of about KES 71,207, closely matching In-store orders (579 orders) at about KES 70,759.
- Filtering for Premium Online orders from Nairobi and Mombasa isolates a focused high-value segment exported to `outputs/premium_online_coast_capital.csv`.

## Setup & Task 1: Load Data and First-Five-Minutes Inspection

In [1]:
import numpy as np
import pandas as pd

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", sales.shape)
print("\n--- HEAD ---")
display(sales.head())
print("\n--- TAIL ---")
display(sales.tail())
print("\n--- INFO ---")
sales.info()
print("\n--- DESCRIBE ---")
display(sales.describe(include="all"))
print("\n--- NUNIQUE ---")
print(sales.nunique())
print("\n--- BRANCH SHARE ---")
print(sales["branch"].value_counts(normalize=True).round(4))
print("\n--- CHANNEL SHARE ---")
print(sales["channel"].value_counts(normalize=True).round(4))
print("\nDate range:", sales["order_date"].min(), "to", sales["order_date"].max())

Shape: (1000, 7)

--- HEAD ---


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500



--- TAIL ---


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200



--- INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB

--- DESCRIBE ---


,order_id,order_date,branch,product,quantity,channel,unit_price
count,1000.000000,1000,1000,1000,1000.000000,1000,1000.000000
unique,NaN,NaN,5,5,NaN,3,NaN
top,NaN,NaN,Nairobi,Laptop,NaN,In-store,NaN
freq,NaN,NaN,376,220,NaN,579,NaN
mean,1500.500000,2026-03-30 08:44:09.600000,NaN,NaN,3.012000,NaN,23918.300000
min,1001.000000,2026-01-01 00:00:00,NaN,NaN,1.000000,NaN,1200.000000
25%,1250.750000,2026-02-14 00:00:00,NaN,NaN,2.000000,NaN,2500.000000
50%,1500.500000,2026-03-29 00:00:00,NaN,NaN,3.000000,NaN,15000.000000
75%,1750.250000,2026-05-14 00:00:00,NaN,NaN,4.000000,NaN,28000.000000
max,2000.000000,2026-06-29 00:00:00,NaN,NaN,5.000000,NaN,65000.000000



--- NUNIQUE ---
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

--- BRANCH SHARE ---
branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64

--- CHANNEL SHARE ---
channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64

Date range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00


### First-Five-Minutes Inspection Notes
- **Shape:** The dataset contains 1,000 rows and 7 columns (`order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, `unit_price`).
- **Head & Tail:** Rows show sequential `order_id` values starting at 1001 with realistic product/price mappings and dates in 2026.
- **Info & Missing Values:** `order_date` is parsed as `datetime64[ns]`, numeric/categorical columns have clean types, and there are zero missing values across columns.
- **Describe & Cardinality:** `quantity` ranges from 1 to 5, `unit_price` spans from KES 1,200 (Charger) to KES 65,000 (Laptop), and there are 5 branches, 5 products, and 3 channels.
- **Date Range & Mix:** Orders span January 1, 2026 through June 29, 2026, led by Nairobi (~40%) and In-store (~55%).

## Task 2: Derived Columns (`revenue`, `month`, `weekday`, `is_weekend`, `price_band`)

In [2]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

for col in ["month", "weekday", "is_weekend", "price_band"]:
    print(f"\n--- value_counts: {col} ---")
    print(sales[col].value_counts())


--- value_counts: month ---
month
March       186
January     173
April       167
June        162
February    159
May         153
Name: count, dtype: int64

--- value_counts: weekday ---
weekday
Friday       166
Sunday       145
Thursday     145
Monday       140
Wednesday    137
Tuesday      135
Saturday     132
Name: count, dtype: int64

--- value_counts: is_weekend ---
is_weekend
False    723
True     277
Name: count, dtype: int64

--- value_counts: price_band ---
price_band
Value      565
Premium    435
Name: count, dtype: int64


/tmp/ipykernel_2970/73077379.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5


## Task 3: Branch Ranking by Revenue vs. Number of Orders

In [3]:
branch_summary = (
    sales.groupby("branch", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_value=("revenue", "mean"),
        units_sold=("quantity", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False, method="min").astype(int)
branch_summary["order_rank"] = branch_summary["order_count"].rank(ascending=False, method="min").astype(int)
branch_summary["revenue_share_pct"] = (branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100).round(1)

display(branch_summary)

,branch,total_revenue,order_count,avg_order_value,units_sold,revenue_rank,order_rank,revenue_share_pct
3,Nairobi,26106100,376,69431.117021,1107,1,1,36.4
2,Mombasa,14003700,186,75288.709677,567,2,2,19.5
4,Nakuru,11970400,162,73891.358025,495,3,3,16.7
1,Kisumu,10868100,159,68352.830189,473,4,4,15.1
0,Eldoret,8844200,117,75591.452991,370,5,5,12.3


**Task 3 Answer:** Ranking branches by total revenue and by order count can differ depending on basket size and high-ticket product mix (such as Laptops and Tablets). A branch with fewer orders can generate higher revenue if customers buy higher-priced items or larger quantities per order.

## Task 4: Top Three Products by Revenue vs. Units Sold

In [4]:
product_summary = (
    sales.groupby("product", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        unit_price=("unit_price", "first"),
    )
)

top3_rev = product_summary.sort_values("total_revenue", ascending=False).head(3)
top3_units = product_summary.sort_values("units_sold", ascending=False).head(3)

print("Top 3 by Revenue:")
display(top3_rev)
print("Top 3 by Units Sold:")
display(top3_units)

Top 3 by Revenue:


,product,total_revenue,units_sold,unit_price
2,Laptop,41535000,639,65000
4,Tablet,19348000,691,28000
3,Phone,8880000,592,15000


Top 3 by Units Sold:


,product,total_revenue,units_sold,unit_price
4,Tablet,19348000,691,28000
2,Laptop,41535000,639,65000
3,Phone,8880000,592,15000


**Task 4 Answer:** Top products by revenue reflect both unit volume and unit price, whereas top products by units sold reflect volume alone. High-unit-price items like Laptops (KES 65,000) and Tablets (KES 28,000) carry heavy revenue weight per unit compared to lower-priced accessories.

## Task 5: Weekend vs. Weekday Orders

In [5]:
weekend_summary = (
    sales.groupby("is_weekend", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
)
weekend_summary["day_type"] = np.where(weekend_summary["is_weekend"], "Weekend", "Weekday")
display(weekend_summary[["day_type", "order_count", "total_revenue", "avg_order_revenue"]])

,day_type,order_count,total_revenue,avg_order_revenue
0,Weekday,723,51856200,71723.651452
1,Weekend,277,19936300,71972.202166


**Task 5 Answer:** Weekdays account for five out of seven days each week, so they naturally generate a larger share of total orders and cumulative revenue, while comparing average order revenue shows whether weekend shoppers spend more or less per transaction.

## Task 6: Best vs. Worst Month for Revenue and Percentage Difference

In [6]:
monthly_rev = (
    sales.assign(year_month=sales["order_date"].dt.to_period("M"))
    .groupby("year_month", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)

best_row = monthly_rev.iloc[0]
worst_row = monthly_rev.iloc[-1]
pct_diff_vs_worst = ((best_row["revenue"] - worst_row["revenue"]) / worst_row["revenue"]) * 100
pct_diff_symmetric = ((best_row["revenue"] - worst_row["revenue"]) / ((best_row["revenue"] + worst_row["revenue"]) / 2)) * 100

display(monthly_rev)
print(f"Best month: {best_row['year_month']} (KES {best_row['revenue']:,.0f})")
print(f"Worst month: {worst_row['year_month']} (KES {worst_row['revenue']:,.0f})")
print(f"Percentage difference (best vs. worst): {pct_diff_vs_worst:.2f}%")
print(f"Percentage difference (relative to midpoint): {pct_diff_symmetric:.2f}%")

,year_month,revenue
3,2026-04,13518300
2,2026-03,12396800
0,2026-01,11786600
1,2026-02,11772000
4,2026-05,11415400
5,2026-06,10903400


Best month: 2026-04 (KES 13,518,300)
Worst month: 2026-06 (KES 10,903,400)
Percentage difference (best vs. worst): 23.98%
Percentage difference (relative to midpoint): 21.41%


**Task 6 Answer:** Comparing the highest-revenue month against the lowest-revenue month highlights seasonal monthly swings across H1 2026, quantified above both as the percentage gain from the lowest month to the highest month and relative to the two-month midpoint.

## Task 7: Filter Premium Online Orders from Nairobi and Mombasa

In [7]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online_coast = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Matching orders count:", len(premium_online_coast))
print("Total revenue: KES", f"{premium_online_coast['revenue'].sum():,.0f}")

premium_online_coast.to_csv("outputs/premium_online_coast_capital.csv", index=False)
display(premium_online_coast.head())

Matching orders count: 83
Total revenue: KES 10,218,000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


**Task 7 Answer:** The filtered subset contains only `Premium` (`unit_price >= 28000`) and `Online` orders originating from `Nairobi` or `Mombasa`, with exact order count and total revenue printed above and exported to `outputs/premium_online_coast_capital.csv` with `index=False`.

## Task 8: Spreadsheet Reconciliation Check (`excel/duka_branch_summary.xlsx`)

In [8]:
branch_channel_pivot = pd.pivot_table(
    sales,
    values="revenue",
    index="branch",
    columns="channel",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)
display(branch_channel_pivot)
print("Grand total revenue in pandas: KES", f"{sales['revenue'].sum():,.0f}")

channel,In-store,Online,Phone order,All
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
All,40969200,22786100,8037200,71792500


Grand total revenue in pandas: KES 71,792,500
